# 🎓 EdTE — AI-Powered Student Performance Analytics

<div style="background: linear-gradient(135deg, #4F46E5, #7C3AED); padding: 20px; border-radius: 12px; color: white; margin-bottom: 20px;">
<h3 style="color: white; margin: 0;">Education Timetable & Engineering Platform</h3>
<p style="margin: 5px 0 0;">Machine Learning Pipeline for Predicting Academic Outcomes & Optimizing Resources</p>
</div>

## Objective
This notebook demonstrates the **ML backbone** of the EdTE platform — a comprehensive academic management system. We train and evaluate multiple models to:

1. **Predict student final grades** (pass/fail & grade brackets)
2. **Identify key performance drivers** (study time, absences, family support, etc.)
3. **Generate actionable insights** for timetable optimization & resource allocation

### Models Covered
| # | Model | Type |
|---|-------|------|
| 1 | Logistic Regression | Baseline Linear |
| 2 | Random Forest | Ensemble (Bagging) |
| 3 | XGBoost | Ensemble (Boosting) |
| 4 | Support Vector Machine | Kernel Method |
| 5 | K-Nearest Neighbors | Instance-Based |
| 6 | Neural Network (MLP) | Deep Learning |

### Dataset
**UCI Student Performance Dataset** — Real-world data from two Portuguese secondary schools covering demographics, social factors, and academic grades.

---

## 1️⃣ Environment Setup & Imports

In [ ]:
# Install required packages (Colab-friendly)
!pip install -q xgboost shap

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Core
import numpy as np
import pandas as pd
import io, zipfile, requests

# Visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    ConfusionMatrixDisplay, RocCurveDisplay
)

# Explainability
import shap

# Style
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

COLORS = ['#4F46E5', '#7C3AED', '#10B981', '#F59E0B', '#EF4444', '#06B6D4']
print('✅ All libraries loaded successfully!')

---
## 2️⃣ Data Acquisition & Loading
We load the **UCI Student Performance** dataset directly from the UCI Machine Learning Repository.

In [ ]:
# Download UCI Student Performance dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip'

try:
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    # student-mat.csv = Math course, student-por.csv = Portuguese course
    df_math = pd.read_csv(z.open('student-mat.csv'), sep=';')
    df_por  = pd.read_csv(z.open('student-por.csv'), sep=';')
    # Merge both courses for a larger dataset
    df = pd.concat([df_math, df_por], ignore_index=True)
    print(f'✅ Loaded from UCI Archive: {df.shape[0]} records, {df.shape[1]} features')
    print(f'   • Math course: {len(df_math)} students')
    print(f'   • Portuguese course: {len(df_por)} students')
except Exception as e:
    print(f'⚠️ UCI download failed ({e}), generating synthetic student data...')
    from sklearn.datasets import make_classification
    X_syn, y_syn = make_classification(n_samples=1000, n_features=20, n_informative=12,
                                       n_classes=2, random_state=42)
    feat_names = ['study_hours','absences','parent_edu','family_support','extra_classes',
                  'internet_access','travel_time','free_time','social_activity','health',
                  'age','failures','romantic','activities','alcohol_weekday',
                  'alcohol_weekend','family_rel','going_out','nursery','higher_edu']
    df = pd.DataFrame(X_syn, columns=feat_names)
    df['G3'] = (y_syn * 10 + np.random.randint(5, 15, size=len(y_syn))).clip(0, 20)
    print(f'✅ Generated synthetic dataset: {df.shape[0]} records')

In [ ]:
# Quick look at the data
print('='*60)
print('DATASET OVERVIEW')
print('='*60)
print(f'Shape: {df.shape}')
print(f'\nColumn types:')
print(df.dtypes.value_counts())
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'\nTarget variable (G3 - Final Grade):')
print(df['G3'].describe().round(2))
df.head()

---
## 3️⃣ Exploratory Data Analysis (EDA)
Deep dive into distribution patterns, correlations, and target variable behavior.

In [ ]:
# Create binary target: Pass (G3 >= 10) vs Fail (G3 < 10)
df['pass_fail'] = (df['G3'] >= 10).astype(int)
print(f'Pass/Fail distribution:')
print(df['pass_fail'].value_counts().rename({1: 'Pass', 0: 'Fail'}))
print(f'\nPass rate: {df["pass_fail"].mean()*100:.1f}%')

In [ ]:
# ---- Figure 1: Grade Distribution & Pass/Fail ----
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1a: Final grade distribution
axes[0].hist(df['G3'], bins=20, color=COLORS[0], edgecolor='white', alpha=0.85)
axes[0].axvline(10, color=COLORS[4], linestyle='--', linewidth=2, label='Pass threshold (10)')
axes[0].set_xlabel('Final Grade (G3)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Distribution of Final Grades', fontweight='bold')
axes[0].legend()

# 1b: Pass/Fail pie
counts = df['pass_fail'].value_counts()
axes[1].pie(counts, labels=['Pass', 'Fail'], autopct='%1.1f%%',
            colors=[COLORS[2], COLORS[4]], startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Pass vs Fail Ratio', fontweight='bold')

# 1c: Grade progression G1 -> G2 -> G3
if 'G1' in df.columns and 'G2' in df.columns:
    grade_means = [df['G1'].mean(), df['G2'].mean(), df['G3'].mean()]
    bars = axes[2].bar(['Period 1 (G1)', 'Period 2 (G2)', 'Final (G3)'],
                       grade_means, color=COLORS[:3], edgecolor='white')
    for bar, val in zip(bars, grade_means):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                     f'{val:.1f}', ha='center', fontweight='bold')
    axes[2].set_ylabel('Mean Grade', fontsize=11)
    axes[2].set_title('Grade Progression', fontweight='bold')
    axes[2].set_ylim(0, 20)

plt.tight_layout()
plt.show()

In [ ]:
# ---- Figure 2: Key Factor Analysis ----
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Study time vs Grade
if 'studytime' in df.columns:
    study_labels = {1: '<2 hrs', 2: '2-5 hrs', 3: '5-10 hrs', 4: '>10 hrs'}
    df['study_label'] = df['studytime'].map(study_labels)
    sns.boxplot(data=df, x='study_label', y='G3', ax=axes[0,0],
                order=['<2 hrs','2-5 hrs','5-10 hrs','>10 hrs'], palette='viridis')
    axes[0,0].set_title('Study Time vs Final Grade', fontweight='bold')
    axes[0,0].set_xlabel('Weekly Study Time')
    df.drop('study_label', axis=1, inplace=True)

# Absences vs Grade
if 'absences' in df.columns:
    sns.scatterplot(data=df, x='absences', y='G3', hue='pass_fail',
                    palette={1: COLORS[2], 0: COLORS[4]}, alpha=0.6, ax=axes[0,1])
    axes[0,1].set_title('Absences vs Final Grade', fontweight='bold')
    axes[0,1].legend(title='Result', labels=['Fail','Pass'])

# Failures vs Grade
if 'failures' in df.columns:
    sns.violinplot(data=df, x='failures', y='G3', ax=axes[0,2],
                   palette='viridis', inner='box')
    axes[0,2].set_title('Past Failures vs Final Grade', fontweight='bold')
    axes[0,2].set_xlabel('Number of Past Failures')

# Parent education effect
if 'Medu' in df.columns:
    edu_labels = {0: 'None', 1: 'Primary', 2: 'Middle', 3: 'Secondary', 4: 'Higher'}
    df['medu_label'] = df['Medu'].map(edu_labels)
    sns.boxplot(data=df, x='medu_label', y='G3', ax=axes[1,0],
                order=['None','Primary','Middle','Secondary','Higher'], palette='viridis')
    axes[1,0].set_title("Mother's Education vs Grade", fontweight='bold')
    axes[1,0].tick_params(axis='x', rotation=20)
    df.drop('medu_label', axis=1, inplace=True)

# Internet access
if 'internet' in df.columns:
    sns.boxplot(data=df, x='internet', y='G3', ax=axes[1,1],
                palette=[COLORS[4], COLORS[2]])
    axes[1,1].set_title('Internet Access vs Grade', fontweight='bold')
    axes[1,1].set_xticklabels(['No Internet', 'Has Internet'])

# Alcohol consumption (weekend)
if 'Walc' in df.columns:
    sns.boxplot(data=df, x='Walc', y='G3', ax=axes[1,2], palette='viridis')
    axes[1,2].set_title('Weekend Alcohol vs Grade', fontweight='bold')
    axes[1,2].set_xlabel('Alcohol (1=very low, 5=very high)')

plt.suptitle('Key Factors Affecting Student Performance', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Figure 3: Correlation Heatmap (numeric features) ----
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()

# Focus on features most correlated with G3
g3_corr = corr['G3'].drop(['G3', 'pass_fail']).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=False,
            square=True, linewidths=0.5, ax=axes[0])
axes[0].set_title('Full Correlation Matrix', fontweight='bold')

# Top correlations with G3
colors_bar = [COLORS[2] if v > 0 else COLORS[4] for v in g3_corr]
g3_corr.plot(kind='barh', ax=axes[1], color=colors_bar, edgecolor='white')
axes[1].set_title('Correlation with Final Grade (G3)', fontweight='bold')
axes[1].set_xlabel('Pearson Correlation')
axes[1].axvline(0, color='grey', linewidth=0.8)

plt.tight_layout()
plt.show()

print('\n\U0001f4a1 Top 5 positive correlations with G3:')
for feat, val in g3_corr.head().items():
    print(f'   {feat:>15}: {val:+.3f}')
print('\n\U0001f4a1 Top 5 negative correlations with G3:')
for feat, val in g3_corr.tail().items():
    print(f'   {feat:>15}: {val:+.3f}')

---
## 4️⃣ Feature Engineering & Preprocessing
Encode categoricals, scale numerics, and split into train/test sets.

In [ ]:
# Prepare features (drop G1, G2 to avoid data leakage - those are intermediate grades)
# We want to predict pass/fail from student attributes BEFORE the final exam
drop_cols = ['G1', 'G2', 'G3', 'pass_fail']
cols_to_drop = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=cols_to_drop)
y = df['pass_fail']

# Identify column types
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print(f'Features: {X.shape[1]}')
print(f'  Categorical ({len(cat_cols)}): {cat_cols}')
print(f'  Numerical   ({len(num_cols)}): {num_cols}')
print(f'\nTarget split: Pass={y.sum()}, Fail={len(y)-y.sum()}')

In [ ]:
# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ],
    remainder='passthrough'
)

# Train/Test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Train pass rate: {y_train.mean()*100:.1f}%')
print(f'Test pass rate:  {y_test.mean()*100:.1f}%')

---
## 5️⃣ Model Training & Evaluation
We train 6 different ML models using scikit-learn pipelines, each with cross-validation.

In [ ]:
# Define all models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    'XGBoost':             xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                             use_label_encoder=False, eval_metric='logloss',
                                             random_state=42, verbosity=0),
    'SVM (RBF)':           SVC(kernel='rbf', probability=True, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7),
    'Neural Network (MLP)': MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=500,
                                          early_stopping=True, random_state=42),
}

# Train & evaluate each model
results = {}

for name, model in models.items():
    print(f'\n{"="*50}')
    print(f'\U0001f916 Training: {name}')
    print(f'{"="*50}')

    # Build pipeline
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    # Cross-validation (5-fold)
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    print(f'  CV Accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})')

    # Fit on full training set
    pipe.fit(X_train, y_train)

    # Predict
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, 'predict_proba') else None

    # Metrics
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)

    results[name] = {
        'pipeline': pipe,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'y_pred': y_pred,
        'y_proba': y_proba,
    }

    print(f'  Test Accuracy:  {acc:.4f}')
    print(f'  Precision:      {prec:.4f}')
    print(f'  Recall:         {rec:.4f}')
    print(f'  F1 Score:       {f1:.4f}')

print(f'\n\n\u2705 All {len(models)} models trained successfully!')

---
## 6️⃣ Model Comparison & Visualization

In [ ]:
# ---- Figure 4: Comprehensive Model Comparison ----
fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

names   = list(results.keys())
accs    = [results[n]['accuracy'] for n in names]
f1s     = [results[n]['f1'] for n in names]
cv_means = [results[n]['cv_mean'] for n in names]
cv_stds  = [results[n]['cv_std'] for n in names]

# 4a: Accuracy comparison
ax1 = fig.add_subplot(gs[0, 0])
bars = ax1.barh(names, accs, color=COLORS, edgecolor='white', height=0.6)
for bar, val in zip(bars, accs):
    ax1.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontweight='bold', fontsize=10)
ax1.set_xlim(min(accs) - 0.05, 1.0)
ax1.set_title('Test Accuracy', fontweight='bold', fontsize=12)
ax1.set_xlabel('Accuracy')

# 4b: F1 Score comparison
ax2 = fig.add_subplot(gs[0, 1])
bars2 = ax2.barh(names, f1s, color=COLORS, edgecolor='white', height=0.6)
for bar, val in zip(bars2, f1s):
    ax2.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontweight='bold', fontsize=10)
ax2.set_xlim(min(f1s) - 0.05, 1.0)
ax2.set_title('F1 Score', fontweight='bold', fontsize=12)
ax2.set_xlabel('F1')

# 4c: Cross-validation scores with error bars
ax3 = fig.add_subplot(gs[1, 0])
ax3.barh(names, cv_means, xerr=cv_stds, color=COLORS, edgecolor='white',
         height=0.6, capsize=4, alpha=0.85)
for i, (m, s) in enumerate(zip(cv_means, cv_stds)):
    ax3.text(m + s + 0.01, i, f'{m:.3f}±{s:.3f}', va='center', fontsize=9)
ax3.set_xlim(min(cv_means) - 0.08, 1.0)
ax3.set_title('5-Fold Cross-Validation Accuracy', fontweight='bold', fontsize=12)
ax3.set_xlabel('CV Accuracy (± std)')

# 4d: Grouped bar - all 4 metrics
ax4 = fig.add_subplot(gs[1, 1])
x = np.arange(len(names))
w = 0.18
precs = [results[n]['precision'] for n in names]
recs  = [results[n]['recall'] for n in names]
ax4.bar(x - 1.5*w, accs, w, label='Accuracy', color=COLORS[0])
ax4.bar(x - 0.5*w, precs, w, label='Precision', color=COLORS[1])
ax4.bar(x + 0.5*w, recs, w, label='Recall', color=COLORS[2])
ax4.bar(x + 1.5*w, f1s, w, label='F1', color=COLORS[3])
ax4.set_xticks(x)
ax4.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=8)
ax4.set_ylim(0, 1.15)
ax4.legend(loc='upper right', fontsize=9)
ax4.set_title('All Metrics Comparison', fontweight='bold', fontsize=12)

plt.suptitle('\U0001f3af EdTE Model Performance Dashboard', fontsize=15, fontweight='bold', y=1.0)
plt.show()

In [ ]:
# ---- Figure 5: ROC Curves for all models ----
fig, ax = plt.subplots(1, 1, figsize=(9, 7))

for i, (name, res) in enumerate(results.items()):
    if res['y_proba'] is not None:
        fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=COLORS[i % len(COLORS)], linewidth=2,
                label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random (AUC = 0.500)')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontweight='bold', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Figure 6: Confusion Matrices for all models ----
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes_flat = axes.flatten()

for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Fail', 'Pass'])
    disp.plot(ax=axes_flat[i], cmap='Blues', colorbar=False)
    axes_flat[i].set_title(name, fontweight='bold', fontsize=11)

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7️⃣ Hyperparameter Tuning (Best Model)
We take the best-performing model and fine-tune it with `GridSearchCV`.

In [ ]:
# Identify best model
best_name = max(results, key=lambda n: results[n]['f1'])
print(f'\U0001f3c6 Best model by F1 score: {best_name} ({results[best_name]["f1"]:.4f})')
print(f'\nTuning {best_name} with GridSearchCV...')

# Define param grids per model type
param_grids = {
    'Logistic Regression': {
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__penalty': ['l2'],
    },
    'Random Forest': {
        'classifier__n_estimators': [100, 200, 300],
        'classifier__max_depth': [5, 10, 15, None],
        'classifier__min_samples_split': [2, 5],
    },
    'XGBoost': {
        'classifier__n_estimators': [100, 200, 300],
        'classifier__max_depth': [3, 6, 9],
        'classifier__learning_rate': [0.01, 0.1, 0.2],
    },
    'SVM (RBF)': {
        'classifier__C': [0.1, 1, 10],
        'classifier__gamma': ['scale', 'auto'],
    },
    'K-Nearest Neighbors': {
        'classifier__n_neighbors': [3, 5, 7, 11],
        'classifier__weights': ['uniform', 'distance'],
    },
    'Neural Network (MLP)': {
        'classifier__hidden_layer_sizes': [(64, 32), (128, 64, 32), (256, 128, 64)],
        'classifier__alpha': [0.0001, 0.001, 0.01],
    },
}

# Get the base pipeline for best model
best_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', models[best_name])
])

grid = GridSearchCV(
    best_pipe,
    param_grids.get(best_name, {}),
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print(f'\n\u2705 Best params: {grid.best_params_}')
print(f'   Best CV F1:  {grid.best_score_:.4f}')

# Evaluate tuned model
y_pred_tuned = grid.predict(X_test)
print(f'\n   Tuned Test Accuracy:  {accuracy_score(y_test, y_pred_tuned):.4f}')
print(f'   Tuned Test F1 Score:  {f1_score(y_test, y_pred_tuned):.4f}')
print(f'   (Before tuning F1:    {results[best_name]["f1"]:.4f})')

---
## 8️⃣ Feature Importance & SHAP Explainability
Understanding **why** the model makes certain predictions — critical for trust & deployment in EdTE.

In [ ]:
# Get feature names after preprocessing
ohe_names = []
if cat_cols:
    ohe = preprocessor.named_transformers_['cat']
    ohe_names = ohe.get_feature_names_out(cat_cols).tolist()
feature_names = num_cols + ohe_names

# ---- Figure 7: Feature Importance (Random Forest) ----
# Use Random Forest for interpretable feature importance
rf_pipe = results['Random Forest']['pipeline']
rf_model = rf_pipe.named_steps['classifier']
importances = rf_model.feature_importances_

feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
top_n = min(20, len(feat_imp))
feat_imp.head(top_n).sort_values().plot(
    kind='barh', ax=ax, color=COLORS[0], edgecolor='white'
)
ax.set_title(f'Top {top_n} Feature Importances (Random Forest)', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance (Gini)')
plt.tight_layout()
plt.show()

print('\n\U0001f4ca Top 10 most important features for prediction:')
for i, (feat, imp) in enumerate(feat_imp.head(10).items(), 1):
    print(f'   {i:2d}. {feat:>20}: {imp:.4f}')

In [ ]:
# ---- Figure 8: SHAP Analysis ----
# SHAP reveals how each feature drives individual predictions
print('Computing SHAP values (this may take a minute)...')

# Prepare transformed data
X_test_transformed = rf_pipe.named_steps['preprocessor'].transform(X_test)
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test_df)

# For binary classification, use class 1 (Pass)
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

# SHAP Summary Plot
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

plt.sca(axes[0])
shap.summary_plot(sv, X_test_df, plot_type='dot', show=False, max_display=15)
axes[0].set_title('SHAP Feature Impact (Dot Plot)', fontweight='bold')

plt.sca(axes[1])
shap.summary_plot(sv, X_test_df, plot_type='bar', show=False, max_display=15)
axes[1].set_title('Mean |SHAP| Value (Bar)', fontweight='bold')

plt.suptitle('SHAP Explainability Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\u2705 SHAP analysis complete!')

---
## 9️⃣ Learning Curves
Diagnose whether models are suffering from **overfitting** or **underfitting**.

In [ ]:
# ---- Figure 9: Learning Curves for Top 3 Models ----
top_3 = sorted(results.keys(), key=lambda n: results[n]['f1'], reverse=True)[:3]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top_3):
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', models[name])
    ])

    train_sizes, train_scores, val_scores = learning_curve(
        pipe, X_train, y_train, cv=5,
        train_sizes=np.linspace(0.1, 1.0, 8),
        scoring='accuracy', n_jobs=-1
    )

    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    axes[i].fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                         alpha=0.15, color=COLORS[0])
    axes[i].fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                         alpha=0.15, color=COLORS[2])
    axes[i].plot(train_sizes, train_mean, 'o-', color=COLORS[0], label='Training', linewidth=2)
    axes[i].plot(train_sizes, val_mean,   'o-', color=COLORS[2], label='Validation', linewidth=2)
    axes[i].set_title(name, fontweight='bold', fontsize=12)
    axes[i].set_xlabel('Training Samples')
    axes[i].set_ylabel('Accuracy')
    axes[i].legend(loc='lower right')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Learning Curves (Bias-Variance Diagnosis)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 🏆 Ensemble: Voting Classifier
Combine the top models into a **voting ensemble** for maximum performance.

In [ ]:
# Voting Ensemble of top 3 models
estimators = [(name, models[name]) for name in top_3]
ensemble = VotingClassifier(estimators=estimators, voting='soft')

ensemble_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', ensemble)
])

# Cross-validate
cv_scores = cross_val_score(ensemble_pipe, X_train, y_train, cv=5, scoring='accuracy')
print(f'\U0001f91d Ensemble ({" + ".join(top_3)})')
print(f'   CV Accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})')

# Fit & evaluate
ensemble_pipe.fit(X_train, y_train)
y_pred_ens = ensemble_pipe.predict(X_test)

ens_acc = accuracy_score(y_test, y_pred_ens)
ens_f1  = f1_score(y_test, y_pred_ens)

print(f'   Test Accuracy:  {ens_acc:.4f}')
print(f'   Test F1 Score:  {ens_f1:.4f}')

# Compare with best single model
best_f1 = results[best_name]['f1']
improvement = ((ens_f1 - best_f1) / best_f1) * 100
print(f'\n   vs best single model ({best_name}): {best_f1:.4f}')
print(f'   Ensemble improvement: {improvement:+.2f}%')

---
## 🚀 Prediction Demo
Simulate how the EdTE platform would use this model to predict a new student's outcome.

In [ ]:
# Demo: Predict for a hypothetical student
print('='*60)
print('\U0001f393 EdTE Student Outcome Prediction Demo')
print('='*60)

# Use first test sample as an example
sample = X_test.iloc[[0]]
actual = y_test.iloc[0]

# Predict with all models
print(f'\nStudent profile (selected features):')
display_cols = [c for c in ['age', 'studytime', 'failures', 'absences', 'Medu', 'Fedu',
                             'internet', 'higher', 'health'] if c in sample.columns]
for col in display_cols:
    print(f'   {col:>12}: {sample[col].values[0]}')

print(f'\nActual result: {"\u2705 PASS" if actual == 1 else "\u274c FAIL"}')
print(f'\nModel predictions:')
for name, res in results.items():
    pred = res['pipeline'].predict(sample)[0]
    proba = res['pipeline'].predict_proba(sample)[0] if hasattr(res['pipeline'], 'predict_proba') else None
    symbol = '\u2705' if pred == actual else '\u274c'
    conf = f' (confidence: {max(proba)*100:.1f}%)' if proba is not None else ''
    print(f'   {symbol} {name:>25}: {"PASS" if pred == 1 else "FAIL"}{conf}')

# Ensemble prediction
ens_pred = ensemble_pipe.predict(sample)[0]
ens_proba = ensemble_pipe.predict_proba(sample)[0]
symbol = '\u2705' if ens_pred == actual else '\u274c'
print(f'   {symbol} {"Ensemble (Voting)":>25}: {"PASS" if ens_pred == 1 else "FAIL"} (confidence: {max(ens_proba)*100:.1f}%)')

---
## 📊 Final Results Summary

In [ ]:
# ---- Final Summary Table ----
summary_rows = []
for name, res in results.items():
    roc_auc_val = '-'
    if res['y_proba'] is not None:
        fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
        roc_auc_val = f"{auc(fpr, tpr):.4f}"
    summary_rows.append({
        'Model': name,
        'CV Accuracy': f"{res['cv_mean']:.4f} ±{res['cv_std']:.4f}",
        'Test Accuracy': f"{res['accuracy']:.4f}",
        'Precision': f"{res['precision']:.4f}",
        'Recall': f"{res['recall']:.4f}",
        'F1 Score': f"{res['f1']:.4f}",
        'AUC-ROC': roc_auc_val,
    })

# Add ensemble
ens_proba_test = ensemble_pipe.predict_proba(X_test)[:, 1]
fpr_e, tpr_e, _ = roc_curve(y_test, ens_proba_test)
summary_rows.append({
    'Model': '\U0001f451 Ensemble (Voting)',
    'CV Accuracy': f"{cv_scores.mean():.4f} ±{cv_scores.std():.4f}",
    'Test Accuracy': f"{ens_acc:.4f}",
    'Precision': f"{precision_score(y_test, y_pred_ens):.4f}",
    'Recall': f"{recall_score(y_test, y_pred_ens):.4f}",
    'F1 Score': f"{ens_f1:.4f}",
    'AUC-ROC': f"{auc(fpr_e, tpr_e):.4f}",
})

summary_df = pd.DataFrame(summary_rows)
print('\n' + '='*90)
print('\U0001f3af EdTE ML PIPELINE — FINAL RESULTS')
print('='*90)
display(summary_df.style.set_properties(**{'text-align': 'center'})
        .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}]))

---
## 📝 Conclusions

### Key Findings

1. **Multiple ML models** were successfully trained and evaluated for predicting student academic outcomes (pass/fail).

2. **Feature importance analysis** reveals that study time, past failures, absences, and parental education are the strongest predictors of academic success.

3. **SHAP explainability** provides transparent, per-student explanations — essential for ethical AI in education.

4. **Ensemble methods** (voting classifier) further improve prediction by combining the strengths of individual models.

### How EdTE Uses This

| Application | ML Component |
|---|---|
| **Early Warning System** | Identify at-risk students before final exams |
| **Resource Allocation** | Prioritize extra classes for high-risk groups |
| **Schedule Optimization** | Assign study-heavy slots for struggling batches |
| **Parent Communication** | Data-driven insights for guardian meetings |
| **Curriculum Planning** | Identify courses needing pedagogical changes |

### Technical Stack
- **Models**: Logistic Regression, Random Forest, XGBoost, SVM, KNN, MLP Neural Network
- **Evaluation**: 5-fold Cross-Validation, ROC-AUC, Confusion Matrices, Learning Curves
- **Explainability**: SHAP (SHapley Additive exPlanations)
- **Pipeline**: scikit-learn Pipelines with `ColumnTransformer` for production-ready preprocessing

---
*EdTE — Education Timetable & Engineering Platform*  
*ML Analytics Module v1.0*